# End-to-End NBA Knowledge Graph Pipeline Validation
This notebook verifies the outputs of the 4-stage pipeline: Information Extraction, KB Construction & Alignment, Reasoning, and RAG.

In [1]:
import os
from rdflib import Graph, Namespace

# Paths
KG_DIR = "../kg_artifacts"
GRAPH_PATH = os.path.join(KG_DIR, "inferred_graph.xml")
ALIGNMENT_PATH = os.path.join(KG_DIR, "alignment.ttl")

print("1. Loading Knowledge Graph")
g = Graph()
g.parse(GRAPH_PATH, format="xml")
print(f"Graph loaded with {len(g)} triples.")

1. Loading Knowledge Graph
Graph loaded with 211077 triples.


In [2]:
print("2. Testing heuristic reasoning (associatedWith)")
query = """
PREFIX nba: <http://example.org/nba/>
SELECT ?player ?team WHERE {
  ?player nba:associatedWith ?team .
} LIMIT 5
"""
results = list(g.query(query))
if len(results) > 0:
    for row in results:
        player = str(row[0]).split('/')[-1]
        team = str(row[1]).split('/')[-1]
        print(f" - {player} is associated with {team}")
else:
    print("No associatedWith relationships found. Run fast_reasoner.py first.")

2. Testing heuristic reasoning (associatedWith)
 - DonovanClingan is associated with theAllNBA
 - Houston is associated with theAllNBA
 - GoldenState is associated with theAllNBA
 - LukeKennard is associated with theAllNBA
 - TariEason is associated with theAllNBA


In [3]:
print("3. Testing wikidata alignment")
align_g = Graph()
try:
    align_g.parse(ALIGNMENT_PATH, format="turtle")
    print(f"Alignment layer loaded with total links: {len(align_g)}")
    
    # Show 3 examples
    for s, p, o in list(align_g)[:3]:
        entity = str(s).split('/')[-1]
        wikidata_id = str(o).split('/')[-1]
        print(f" - {entity} mapped to Wikidata ID: {wikidata_id}")
except Exception as e:
    print("Could not load alignment.ttl")

3. Testing wikidata alignment
Alignment layer loaded with total links: 980
 - PortlandTrailBlazers mapped to Wikidata ID: Q167253
 - Charania mapped to Wikidata ID: Q37311856
 - DavionMitchell mapped to Wikidata ID: Q100885338


In [4]:
print("4. Verifying KGE Dataset Splits...")
for split in ["train.txt", "valid.txt", "test.txt"]:
    path = os.path.join("../data/kge", split) # Adjust if your KGE files are in src/kge/
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024
        print(f" ✅ {split} found ({size:.2f} KB)")
    else:
        print(f" ❌ {split} missing!")

4. Verifying KGE Dataset Splits...
 ✅ train.txt found (5522.41 KB)
 ✅ valid.txt found (690.51 KB)
 ✅ test.txt found (694.80 KB)
